In [0]:
CREATE DATABASE IF NOT EXISTS nyc_taxi;

-- =================INITIALIZE BRONZE SCHEMA=====================

--bronze_payment
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_payment (
    payment_type_code STRING,
    payment_type STRING,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/payment'
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');

--bronze_zone
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_zone (
    LocationID STRING,
    Borough STRING,
    Zone STRING,
    service_zone STRING,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/zone'
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');


--bronze_type
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_type(
    trip_type STRING,
    description STRING,
    file_name STRING,
    created_on TIMESTAMP
) USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/type'
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');


--bronze_green_trip
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze_green_trip (
    VendorID INT,
    lpep_pickup_datetime TIMESTAMP_NTZ,
    lpep_dropoff_datetime TIMESTAMP_NTZ,
    store_and_fwd_flag STRING,
    RatecodeID BIGINT,
    PULocationID INT,
    DOLocationID INT,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    ehail_fee DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    payment_type BIGINT,
    trip_type BIGINT,
    congestion_surcharge DOUBLE,
    year INT,
    month INT,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/bronze/green_taxi'
PARTITIONED BY (year, month)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')

;

-- =================INITIALIZE SILVER SCHEMA=====================
--silver_green_trip

CREATE TABLE IF NOT EXISTS nyc_taxi.silver_green_trip (
    deterministic_hash_key STRING,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    pickup_zone_id INT,
    dropoff_zone_id INT,
    payment_id INT,
    trip_type_id INT,
    passenger_count INT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    total_amount DOUBLE,
    year INT,
    month INT,
    trip_duration DOUBLE,
    average_speed DOUBLE,
    extra_charge DOUBLE,
    created_on TIMESTAMP,
    modified_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/silver/green_taxi'
PARTITIONED BY (year, month)
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');

--silver_type

CREATE TABLE IF NOT EXISTS nyc_taxi.silver_type (
    trip_type_id INT,
    trip_type STRING,
    created_on TIMESTAMP,
    modified_on TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/silver/type'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

-- silver_zone
CREATE TABLE IF NOT EXISTS nyc_taxi.silver_zone (
    zone_id INT,
    borough STRING,
    zone1 STRING,
    zone2 STRING,
    service_zone STRING,
    created_on TIMESTAMP,
    modified_on TIMESTAMP
)
USING DELTA
LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/silver/zone'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

-- silver_payment
CREATE TABLE IF NOT EXISTS nyc_taxi.silver_payment(
    payment_type_id INT,
    payment_type STRING,
    created_on TIMESTAMP,
    modified_on TIMESTAMP
) USING DELTA 
LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/silver/payment'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

-- =================INITIALIZE GOLD SCHEMA=====================

CREATE TABLE IF NOT EXISTS nyc_taxi.dim_payment(
    payment_type_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    payment_type_id INTEGER,
    payment_type STRING
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/gold/payment';

CREATE TABLE IF NOT EXISTS nyc_taxi.dim_type(
    trip_type_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    trip_type_id INTEGER,
    trip_type STRING
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/gold/type';

CREATE TABLE IF NOT EXISTS nyc_taxi.dim_zone (
    zone_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    zone_id INTEGER,
    borough STRING,
    service_zone STRING,
    zone1 STRING,
    zone2 STRING
)
USING DELTA LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/gold/zone';

-- fact_trips
CREATE TABLE IF NOT EXISTS nyc_taxi.fact_trips (
    trip_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    trip_id STRING,
    pickup_zone_sk LONG,
    dropoff_zone_sk LONG,
    trip_type_sk LONG,
    payment_type_sk LONG,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count INT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    total_amount DOUBLE,
    extra_charge DOUBLE,
    trip_duration DOUBLE,
    average_speed DOUBLE,
    year INT,
    month INT
)
USING DELTA
LOCATION 'abfss://nyc-taxi@gen2nyctaxi.dfs.core.windows.net/gold/fact_trips'
PARTITIONED BY (year, month)
